In [1]:
from unc_handling import UG_prompter
from DataLoader import DataLoader
from segmentation import Segmentation
from segmentation_util import combine_prompt_sets
from evaluation import Evaluator, compare_to_recontours, save_evaluation_results, evaluate_slice_by_slice
from pathlib import Path
import numpy as np
import pandas as pd


root = r"C:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\data\LUNDPROBE\ExtendedSamples\development"

methods_available = ["raycast", "local_normals"]
propagation_styles = ['default', 'full', 'prompt_based', 'central_start', 'central_partitions']
method = methods_available[1]
propagation_style = propagation_styles[3]

rootpath = Path(root)
subjects = sorted([p.name for p in rootpath.iterdir() if p.is_dir()])
print(subjects)

['newAcq_050f229dc2bdb64c', 'newAcq_0b4940fa31a1d650', 'newAcq_0cc559a8bd82a14a', 'newAcq_1b911d6cb2348f30', 'newAcq_1e0f8b9b01ce5f0b', 'newAcq_250d6075dd465a1a', 'newAcq_433a8d44fddd5b7f', 'newAcq_47ceabdbca398517', 'newAcq_486b7494ee9d71e7', 'newAcq_4a136e8fe320bd13']


In [7]:
from pathlib import Path
import numpy as np
import pandas as pd


def evaluate_saved_logit_fusion(
    root,
    subjects,
    weights=(0.5, 0.5),
    logits_folder="saved_logits",
    output_folder="fusion_evaluation",
    threshold=0.0,
    surface_dice_tol=1.0,
):
    """
    Create segmentations from weighted saved logit maps and evaluate them
    both per volume and per slice.

    Assumes each .npz file contains:
        - "Dense_and_nietjes"
        - "Uncertainty_bboxes"
    """

    if len(weights) != 2:
        raise ValueError("weights must contain exactly two values.")

    weight_dense, weight_bbox = weights

    if weight_dense < 0 or weight_bbox < 0:
        raise ValueError("Weights cannot be negative.")

    if weight_dense == 0 and weight_bbox == 0:
        raise ValueError("At least one weight must be greater than zero.")

    logits_folder = Path(logits_folder)
    output_folder = Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)

    volume_results = []
    slice_results = []

    for subject_nr in range(len(subjects)):
        # Load image data, ground truth, spacing, uncertainty, etc.
        data = DataLoader(
            parentfolder=root,
            subject_nr=subject_nr,
            volume_of_interest="CTVT",
            verbose=True,
        )

        data.load_recontours()
        data.load_consensus()

        subject_name = data.subject_name

        logits_path = (
            logits_folder /
            f"{subject_name}_logits.npz"
        )

        if not logits_path.exists():
            print(f"Skipping {subject_name}: file not found:\n{logits_path}")
            continue

        # The context manager closes the .npz file after loading.
        with np.load(logits_path) as saved_logits:
            print(f"Available arrays: {saved_logits.files}")

            dense_logits = saved_logits[
                "Dense_and_nietjes"
            ].astype(np.float32)

            bbox_logits = saved_logits[
                "Uncertainty_bboxes"
            ].astype(np.float32)

        if dense_logits.shape != bbox_logits.shape:
            raise ValueError(
                f"Logit shapes differ for {subject_name}: "
                f"{dense_logits.shape} versus {bbox_logits.shape}."
            )

        if dense_logits.shape != data.gt.shape:
            raise ValueError(
                f"Logit and ground-truth shapes differ for {subject_name}: "
                f"logits={dense_logits.shape}, gt={data.gt.shape}."
            )

        # Weighted logit fusion
        fused_logits = (
            weight_dense * dense_logits
            + weight_bbox * bbox_logits
        )

        # Convert fused logits to a binary segmentation
        fused_seg = fused_logits > threshold

        if not fused_seg.any():
            print(
                f"Skipping evaluation for {subject_name}: "
                "the fused segmentation is empty."
            )
            continue

        # ----------------------------------------------------
        # Whole-volume evaluation
        # ----------------------------------------------------
        volume_evaluator = Evaluator(
            pred=fused_seg,
            gt=data.consensus_recontour,
            spacing=data.img_spacing,
            subject_name=subject_name,
        )

        volume_metrics = volume_evaluator.compute_all(
            surface_dice_tol=surface_dice_tol
        )

        volume_metrics["StapleWeight"] = weight_dense
        volume_metrics["BBoxWeight"] = weight_bbox
        volume_metrics["LogitThreshold"] = threshold

        volume_results.append(volume_metrics)

        # ----------------------------------------------------
        # Slice-wise evaluation
        # ----------------------------------------------------
        subject_slice_results = evaluate_slice_by_slice(
            pred=fused_seg,
            gt=data.consensus_recontour,
            uncertainty=data.unc_map,
            spacing=data.img_spacing,
            subject_name=subject_name,
            surface_dice_tol=surface_dice_tol,
        )

        subject_slice_results["StapleWeight"] = weight_dense
        subject_slice_results["BBoxWeight"] = weight_bbox
        subject_slice_results["LogitThreshold"] = threshold

        slice_results.append(subject_slice_results)
        print(f"Evaluated and saved fusion for {subject_name}")

    # --------------------------------------------------------
    # Combine and save results
    # --------------------------------------------------------
    volume_df = pd.DataFrame(volume_results)

    if slice_results:
        slice_df = pd.concat(slice_results, ignore_index=True)
    else:
        slice_df = pd.DataFrame()

    weight_name = f"Staple_{weight_dense}_bbox_{weight_bbox}"

    volume_csv = output_folder / f"volume_results_{weight_name}.csv"
    slice_csv = output_folder / f"slice_results_{weight_name}.csv"

    volume_df.to_csv(volume_csv, index=False)
    slice_df.to_csv(slice_csv, index=False)

    print(f"\nSaved volume evaluation to:\n{volume_csv}")
    print(f"\nSaved slice-wise evaluation to:\n{slice_csv}")

    return volume_df, slice_df

In [8]:
weighting_list = [
    (0.0, 1.0),
    (0.1, 0.9),
    (0.2, 0.8),
    (0.3, 0.7),
    (0.4, 0.6),
    (0.5, 0.5),
    (0.6, 0.4),
    (0.7, 0.3),
    (0.8, 0.2),
    (0.9, 0.1),
    (1.0, 0.0),
]

for weights in weighting_list:
    evaluate_saved_logit_fusion(
        root=root,
        subjects=subjects,
        weights=weights,
        logits_folder="saved_logits",
        output_folder="fusion_evaluation",
        threshold=0.0,
        surface_dice_tol=1.0,
    )

Loaded subject newAcq_050f229dc2bdb64c with volume of interest 'CTVT'
Image shape: (88, 1024, 1024), Mask shape: (88, 1024, 1024), Uncertainty map shape: (88, 1024, 1024), Ground truth shape: (88, 1024, 1024)
Image spacing (z, y, x): (2.5, 0.46880000829696655, 0.46880000829696655)
Available arrays: ['Dense_and_nietjes', 'Uncertainty_bboxes']
Evaluating slices 27 to 44...
Number of evaluated slices: 18
Evaluated and saved fusion for newAcq_050f229dc2bdb64c
Loaded subject newAcq_0b4940fa31a1d650 with volume of interest 'CTVT'
Image shape: (88, 1024, 1024), Mask shape: (88, 1024, 1024), Uncertainty map shape: (88, 1024, 1024), Ground truth shape: (88, 1024, 1024)
Image spacing (z, y, x): (2.5, 0.46880000829696655, 0.46880000829696655)
Available arrays: ['Dense_and_nietjes', 'Uncertainty_bboxes']
Evaluating slices 23 to 50...
Number of evaluated slices: 28
Evaluated and saved fusion for newAcq_0b4940fa31a1d650
Loaded subject newAcq_0cc559a8bd82a14a with volume of interest 'CTVT'
Image shap

In [5]:
from pathlib import Path
import pandas as pd


def evaluate_existing_masks(
    root,
    subjects,
    output_folder="existing_mask_evaluation",
    surface_dice_tol=1.0,
):
    """
    Evaluate nnUNet and each observer against the consensus.

    Saves separate volume-wise and slice-wise CSV files for:
        - nnUnet
        - obsB
        - obsC
        - obsD
        - obsE
    """

    output_folder = Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)

    result_names = ["nnUnet", "obsB", "obsC", "obsD", "obsE"]

    volume_results = {
        name: []
        for name in result_names
    }

    slice_results = {
        name: []
        for name in result_names
    }

    for subject_nr in range(len(subjects)):

        data = DataLoader(
            parentfolder=root,
            subject_nr=subject_nr,
            volume_of_interest="CTVT",
            verbose=True,
        )

        data.load_recontours()
        data.load_consensus()

        subject_name = data.subject_name

        # ====================================================
        # Masks to evaluate against the consensus
        # ====================================================

        masks_to_evaluate = {
            "nnUnet": data.mask,
            "obsB": data.observer_recontours[0],
            "obsC": data.observer_recontours[1],
            "obsD": data.observer_recontours[2],
            "obsE": data.observer_recontours[3],
        }

        for result_name, pred in masks_to_evaluate.items():

            # ================================================
            # Whole-volume evaluation
            # ================================================

            volume_evaluator = Evaluator(
                pred=pred,
                gt=data.consensus_recontour,
                spacing=data.img_spacing,
                subject_name=subject_name,
            )

            volume_metrics = volume_evaluator.compute_all(
                surface_dice_tol=surface_dice_tol
            )

            volume_results[result_name].append(volume_metrics)

            # ================================================
            # Slice-wise evaluation
            # ================================================

            subject_slice_results = evaluate_slice_by_slice(
                pred=pred,
                gt=data.consensus_recontour,
                uncertainty=data.unc_map,
                spacing=data.img_spacing,
                subject_name=subject_name,
                surface_dice_tol=surface_dice_tol,
            )

            slice_results[result_name].append(
                subject_slice_results
            )

        print(f"Evaluated {subject_name}")

    # ========================================================
    # Combine and save each evaluation separately
    # ========================================================

    saved_volume_dfs = {}
    saved_slice_dfs = {}

    for result_name in result_names:

        volume_df = pd.DataFrame(
            volume_results[result_name]
        )

        if slice_results[result_name]:
            slice_df = pd.concat(
                slice_results[result_name],
                ignore_index=True,
            )
        else:
            slice_df = pd.DataFrame()

        volume_csv = output_folder / f"{result_name}_volume.csv"
        slice_csv = output_folder / f"{result_name}_slice.csv"

        volume_df.to_csv(volume_csv, index=False)
        slice_df.to_csv(slice_csv, index=False)

        saved_volume_dfs[result_name] = volume_df
        saved_slice_dfs[result_name] = slice_df

        print(f"Saved {volume_csv}")
        print(f"Saved {slice_csv}")

    return saved_volume_dfs, saved_slice_dfs

In [6]:
evaluate_existing_masks(
    root=root,
    subjects=subjects,
    output_folder="existing_mask_evaluation",
    surface_dice_tol=1.0,
)

Loaded subject newAcq_050f229dc2bdb64c with volume of interest 'CTVT'
Image shape: (88, 1024, 1024), Mask shape: (88, 1024, 1024), Uncertainty map shape: (88, 1024, 1024), Ground truth shape: (88, 1024, 1024)
Image spacing (z, y, x): (2.5, 0.46880000829696655, 0.46880000829696655)
Evaluating slices 27 to 44...
Number of evaluated slices: 18
Evaluating slices 28 to 44...
Number of evaluated slices: 17
Evaluating slices 27 to 44...
Number of evaluated slices: 18
Evaluating slices 27 to 44...
Number of evaluated slices: 18
Evaluating slices 27 to 44...
Number of evaluated slices: 18
Evaluated newAcq_050f229dc2bdb64c
Loaded subject newAcq_0b4940fa31a1d650 with volume of interest 'CTVT'
Image shape: (88, 1024, 1024), Mask shape: (88, 1024, 1024), Uncertainty map shape: (88, 1024, 1024), Ground truth shape: (88, 1024, 1024)
Image spacing (z, y, x): (2.5, 0.46880000829696655, 0.46880000829696655)
Evaluating slices 23 to 50...
Number of evaluated slices: 28
Evaluating slices 23 to 51...
Number

({'nnUnet':               subject name      HD_mm   HD95_mm    MSD_mm   ASSD_mm      Dice  \
  0  newAcq_050f229dc2bdb64c   2.710879  0.468800  0.133438  0.138528  0.983942   
  1  newAcq_0b4940fa31a1d650  14.311951  5.575339  0.868842  0.867263  0.921616   
  2  newAcq_0cc559a8bd82a14a   2.868442  0.468800  0.116583  0.110519  0.987830   
  3  newAcq_1b911d6cb2348f30   2.500000  0.468800  0.085652  0.084942  0.984526   
  4  newAcq_1e0f8b9b01ce5f0b   2.500000  0.468800  0.100737  0.096573  0.984465   
  5  newAcq_250d6075dd465a1a   2.543575  0.468800  0.085390  0.091777  0.989201   
  6  newAcq_433a8d44fddd5b7f   3.017790  0.468800  0.085723  0.082762  0.983472   
  7  newAcq_47ceabdbca398517   5.215145  1.994329  0.210724  0.278562  0.970894   
  8  newAcq_486b7494ee9d71e7   2.227124  0.498000  0.077482  0.080250  0.983221   
  9  newAcq_4a136e8fe320bd13   5.000000  2.096537  0.264631  0.273196  0.972582   
  
     SurfaceDice@1.0mm  CentroidDistance_mm  PredictionVolume_mm3  \
  0  